# 07 Final preprocessing

Rebuild the approved offline feature frame from the cleaned release and the frozen split. The [leakage audit](../reports/leakage_audit.md) traces all 16 feature inputs. This run is retrospective because label confirmation times and live data arrival times are unavailable.

Fit the transformer on training features alone. Use the frozen validation and test features only for transformation. Keep all labels separate, apply no resampling, and compute optional class weights using training labels only.

## Transformation choices

- **Numerical inputs.** Impute with training medians, then standardize the continuous and ordered fields, including the elapsed times, counts, rates and leg. A median is less affected by the tails observed in the training EDA. Missing history on a first event still has a separate signal because its prior-event count is zero.
- **Categorical inputs.** Treat the event type and three binary flags as categories. Use the training mode if an input is missing. Encode categories with `OneHotEncoder(handle_unknown="ignore")`. A new category contributes zeros to that field's encoded columns.
- **Memory.** Use 32-bit floats and sparse one-hot encoding. The transformer keeps the combined result sparse when fewer than half of its cells are populated according to its stacking rule. Above that threshold a compact dense array avoids storing an index alongside most values. Measure the representation and size after transformation.
- **Class imbalance.** Use the inverse training-class-frequency formula as an initial weighting option. Weights are saved for later model fitting. This notebook fits no classifier.

[Training EDA](../reports/eda_findings.md) · [Verified feature definitions](06_Feature_Engineering.ipynb)

In [ ]:
from hashlib import file_digest
from pathlib import Path
from tempfile import TemporaryDirectory
import json
import sys

import joblib
import numpy as np
import pandas as pd
from scipy import sparse
import sklearn
import yaml
from IPython.display import display

current = Path.cwd()
root = next(path for path in (current, *current.parents) if (path / "AGENTS.md").is_file())
sys.path.insert(0, str(root))
sys.dont_write_bytecode = True
from src.features import FEATURE_COLUMNS, build_features
from src.integration import (
    BATCH_READ_COLUMNS, ENTITY_READ_COLUMNS, MEDICINE_READ_COLUMNS, PREDICTION_TIMESTAMP,
    TRANSACTION_KEY, TRANSACTION_READ_COLUMNS, integrate_transactions, require_unique_key,
)
from src.preprocessing import (
    CATEGORICAL_FEATURES, NUMERICAL_FEATURES, SPLIT_NAMES, TARGET_COLUMN,
    align_target, index_targets, make_preprocessor, training_class_weights,
)

processed = root / "processed"
cleaned = processed / "cleaned"
artifacts = root / "artifacts"
boundaries = json.loads((processed / "split_boundaries.json").read_text(encoding="utf-8"))
manifest_path = processed / boundaries["assignment_file"]
target_path = processed / "transaction_target.parquet"
key = list(TRANSACTION_KEY)
clock = PREDICTION_TIMESTAMP
assert key == boundaries["transaction_key"] and clock == boundaries["prediction_timestamp"]
assert TARGET_COLUMN == boundaries["target_column"]

def sha256(path):
    with path.open("rb") as source:
        return file_digest(source, "sha256").hexdigest()

assert sha256(manifest_path) == boundaries["assignment_sha256"]
assert sha256(target_path) == boundaries["input_sha256"][target_path.relative_to(root).as_posix()]
report = (root / "reports/leakage_audit.md").read_text(encoding="utf-8")
assert all(f"| `{name}` |" in report for name in FEATURE_COLUMNS)
assert set(FEATURE_COLUMNS).isdisjoint({"batch_id", clock, "split", TARGET_COLUMN})
assert len(NUMERICAL_FEATURES) + len(CATEGORICAL_FEATURES) == len(FEATURE_COLUMNS)
print(f"Verified {len(FEATURE_COLUMNS)} audited feature inputs and the frozen artifacts. Scikit-learn {sklearn.__version__}.")

## Rebuild the split features

Use the same deterministic integration and strictly earlier batch history as in the previous notebooks. Match the manifest by composite key and timestamp. Keep the keys for later target alignment, and select only audited columns for `X`. The validation and test features follow the same fixed rules. No label values are read in this step.

In [ ]:
def read_cleaned(name, columns):
    return pd.read_parquet(cleaned / f"{name}.parquet", columns=list(columns))


config = yaml.safe_load((root / "Phase1_Dataset/code/config.yaml").read_text(encoding="utf-8"))
integrated, _, timing = integrate_transactions(
    read_cleaned("transactions", TRANSACTION_READ_COLUMNS),
    read_cleaned("batches", BATCH_READ_COLUMNS),
    read_cleaned("medicines", MEDICINE_READ_COLUMNS),
    read_cleaned("entities", ENTITY_READ_COLUMNS),
    config["storage_ranges"],
)
assert timing["storage_rate_matches"] == len(integrated)
identity = [*key, clock]
manifest = pd.read_parquet(manifest_path, columns=[*identity, "split"])
require_unique_key(manifest, key, "split manifest")
assigned = integrated.merge(manifest, on=identity, how="inner", validate="one_to_one")
assert len(assigned) == len(manifest) == len(integrated) == boundaries["source_rows"]
assert assigned.groupby("batch_id")["split"].nunique().eq(1).all()
assert set(assigned["split"]) == {*SPLIT_NAMES, boundaries["excluded_assignment"]}
del manifest, integrated

retained = assigned.loc[assigned["split"].isin(SPLIT_NAMES)]
assert len(retained) == boundaries["retained_rows"]
del assigned
engineered = build_features(retained)
chronological = retained[[*identity, "split"]].sort_values([clock, *key], ignore_index=True)
pd.testing.assert_frame_equal(engineered[identity], chronological[identity])
del retained

raw_features = {}
row_keys = {}
for split in SPLIT_NAMES:
    selected = chronological["split"].eq(split)
    raw_features[split] = engineered.loc[selected, list(FEATURE_COLUMNS)].reset_index(drop=True)
    row_keys[split] = chronological.loc[selected, key].reset_index(drop=True)
    assert len(raw_features[split]) == len(row_keys[split]) == boundaries["split_summary"][split]["rows"]
    assert list(raw_features[split].columns) == list(FEATURE_COLUMNS)
del engineered, chronological

X_train_raw, X_validation_raw, X_test_raw = (raw_features[name] for name in SPLIT_NAMES)
print("Separated the three frozen feature sets without loading label values.")

## Fit on training and transform each split

Fit one `ColumnTransformer` using only `X_train_raw`. Check the fitted medians, modes, category dictionaries and scaler against training data. Apply it to the two other feature sets with `transform` only. Compare the fitted transformer's fingerprint before and after those calls.

In [ ]:
preprocessor = make_preprocessor()
X_train = preprocessor.fit_transform(X_train_raw)

numeric = preprocessor.named_transformers_["numeric"]
training_numeric = X_train_raw.loc[:, list(NUMERICAL_FEATURES)].to_numpy(dtype=np.float32, na_value=np.nan)
np.testing.assert_allclose(
    numeric["impute"].statistics_, np.nanmedian(training_numeric, axis=0), rtol=1e-6,
)
imputed_training = numeric["impute"].transform(training_numeric)
np.testing.assert_allclose(numeric["scale"].mean_, imputed_training.mean(axis=0, dtype=np.float64), rtol=1e-10)
assert np.all(numeric["scale"].n_samples_seen_ == len(X_train_raw))
del training_numeric, imputed_training

categorical = preprocessor.named_transformers_["categorical"]
encoder = categorical["encode"]
for name, mode, learnt in zip(CATEGORICAL_FEATURES, categorical["impute"].statistics_, encoder.categories_, strict=True):
    values = X_train_raw[name].astype("string")
    assert mode in values.mode().tolist()
    assert set(learnt) == set(values.dropna().unique())
assert encoder.handle_unknown == "ignore" and encoder.sparse_output
feature_names = preprocessor.get_feature_names_out().tolist()
assert len(feature_names) == len(set(feature_names)) == X_train.shape[1]

fitted_hash = joblib.hash(preprocessor)
X_validation = preprocessor.transform(X_validation_raw)
X_test = preprocessor.transform(X_test_raw)
assert joblib.hash(preprocessor) == fitted_hash
print("Training medians, scaler, modes and categories verified. Validation and test left the fitted state unchanged.")

## Align the separate target and calculate training weights

Read the saved binary labels after the transformation has been fitted. Align each split's `y` to its feature rows by the full transaction key. Calculate class weights as training row count divided by twice the frequency of the class. Validation and test labels are kept separately for later evaluation.

In [ ]:
target_table = pd.read_parquet(target_path, columns=[*key, TARGET_COLUMN])
assert len(target_table) == boundaries["source_rows"]
labels_by_key = index_targets(target_table)
y_train, y_validation, y_test = (align_target(row_keys[name], labels_by_key) for name in SPLIT_NAMES)
del target_table, labels_by_key
class_weights = training_class_weights(y_train)
training_counts = np.bincount(y_train, minlength=2)
assert sum(training_counts) == len(X_train_raw)
np.testing.assert_allclose(
    [class_weights[0], class_weights[1]], len(y_train) / (2 * training_counts),
)
display(pd.DataFrame({"training_count": training_counts, "weight": [class_weights[0], class_weights[1]]},
                     index=pd.Index([0, 1], name="class")))

## Verify output dimensions, missing values, and storage

Each split must keep its original row count and receive the same output columns. Check that the encoded matrices contain finite 32-bit values. Compare the actual storage with an estimate for an equivalent sparse matrix. Test an unseen event type and a missing event type with the already fitted transformer.

In [ ]:
results = []
for name, matrix, labels in zip(
    SPLIT_NAMES, (X_train, X_validation, X_test), (y_train, y_validation, y_test), strict=True,
):
    expected_rows = boundaries["split_summary"][name]["rows"]
    assert matrix.shape == (expected_rows, len(feature_names)) and labels.shape == (expected_rows,)
    assert matrix.dtype == np.float32 and labels.dtype == np.uint8
    if sparse.issparse(matrix):
        assert sparse.isspmatrix_csr(matrix)
        values = matrix.data
        storage = values.nbytes + matrix.indices.nbytes + matrix.indptr.nbytes
        nonzero = int(np.count_nonzero(values))
        representation = "CSR"
    else:
        values = matrix
        storage = values.nbytes
        nonzero = int(np.count_nonzero(values))
        representation = "dense"
    assert np.isfinite(values).all()
    estimated_csr = nonzero * 8 + (expected_rows + 1) * 4
    results.append({
        "split": name, "X shape": matrix.shape, "y shape": labels.shape, "representation": representation,
        "nonzero pct": 100 * nonzero / (expected_rows * len(feature_names)),
        "storage MiB": storage / 2**20, "estimated CSR MiB": estimated_csr / 2**20,
    })

def as_dense(matrix):
    return matrix.toarray() if sparse.issparse(matrix) else matrix


example = X_train_raw.iloc[:1]
event_columns = [i for i, name in enumerate(feature_names) if name.startswith("categorical__event_type_")]
assert len(event_columns) == len(encoder.categories_[0])
unseen = as_dense(preprocessor.transform(example.assign(event_type="__unseen_for_validation_check__")))
missing = as_dense(preprocessor.transform(example.assign(event_type=pd.NA)))
mode = as_dense(preprocessor.transform(example.assign(event_type=categorical["impute"].statistics_[0])))
assert np.count_nonzero(unseen[:, event_columns]) == 0
np.testing.assert_array_equal(missing[:, event_columns], mode[:, event_columns])
assert joblib.hash(preprocessor) == fitted_hash
summary = pd.DataFrame(results).set_index("split")
display(summary.round(2))

## Save reusable artifacts

Save the fitted transformer, the output column names in exact matrix order, and the training class weights. Reload the transformer and verify that it produces exactly the same result as the original on a training example. The split matrices and targets remain in memory.

In [ ]:
def save_json(path, value):
    path.write_text(json.dumps(value, indent=2, allow_nan=False) + "\n", encoding="utf-8")


artifacts.mkdir(exist_ok=True)
with TemporaryDirectory(dir=artifacts) as folder:
    staging = Path(folder)
    fitted_path = staging / "preprocessor.joblib"
    names_path = staging / "final_feature_names.json"
    weights_path = staging / "training_class_weights.json"
    joblib.dump(preprocessor, fitted_path)
    save_json(names_path, feature_names)
    save_json(weights_path, {
        "label_column": TARGET_COLUMN,
        "training_rows": len(y_train),
        "training_class_counts": {str(label): int(count) for label, count in enumerate(training_counts)},
        "class_weights": class_weights,
        "split_assignment_sha256": boundaries["assignment_sha256"],
    })

    restored = joblib.load(fitted_path)
    assert restored.get_feature_names_out().tolist() == json.loads(names_path.read_text(encoding="utf-8"))
    np.testing.assert_array_equal(
        as_dense(restored.transform(example)), as_dense(preprocessor.transform(example)),
    )
    stored_weights = json.loads(weights_path.read_text(encoding="utf-8"))["class_weights"]
    assert {int(label): value for label, value in stored_weights.items()} == class_weights
    for path in (fitted_path, names_path, weights_path):
        path.replace(artifacts / path.name)

print("Saved the verified preprocessor, output feature names, and training class weights. All assertions passed.")